# Week 3.2: Testing Your Twin
### Plain-English edition — no coding required

> ▶ **Do this first: click _Connect_ at the top right, now.** Colab takes 10-30 seconds to
> start a computer for you. Let it happen while you read - by the time you reach Step 1
> it will be ready, and every step will run in about a second.

> **How do you write unit tests for something that never gives the same answer twice?**

Ordinary software is tested by checking that `2 + 2` returns exactly `4`. A twin might say
*"The answer is 4"*, or *"That would be four"*, or *"4!"* — all correct, none identical.

So AI testing asks a looser question: **does the reply contain what it must, and avoid what
it must not?** Get that right and you can change your twin with confidence, because you will
know immediately if you broke something.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **Why you cannot eyeball it** — bad answers look exactly like good ones
- **A test case** — a question, what you expect, and how strictly to judge it
- **Four ways to judge** — exact, contains, all-words, and must-not-contain
- **A test suite** — many tests, one score
- **Catching a regression** — the moment testing pays for itself
- **Measuring quality** — relevance, and answers not supported by the facts
- **Testing a real model** — optional, with your Groq key

---
## Part 1: Why you cannot just read the answers

Here are three versions of the same twin. One is correct. One has quietly broken. One has
stopped being useful while sounding delightful. Read their replies and try to spot which.

In [1]:
#@title ▶ Step 1 — Can you spot the broken twin by eye? { display-mode: "form" }
question = "Who made you?" #@param ["Who made you?", "What is 2 + 2?", "What is the weather today?", "When does the pharmacy open?"] {allow-input: true}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

TWINS = {
 "version 1 (the one that works)": {
   "2 + 2": "The answer is 4.",
   "who made you": "I am a digital twin, built by Sami.",
   "weather": "I do not have access to live weather, so I cannot say.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 2 (after a careless change)": {
   "2 + 2": "I think it is around 4 or 5.",
   "who made you": "I am ChatGPT, a large language model.",
   "weather": "It is sunny and 28 degrees today.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 3 (polite but useless)": {
   "2 + 2": "That is a great question! Let me think about it.",
   "who made you": "That is a great question! Let me think about it.",
   "weather": "That is a great question! Let me think about it.",
   "pharmacy": "That is a great question! Let me think about it.",
   "5km": "That is a great question! Let me think about it.",
 },
}
def twin_reply(question, version):
    table = TWINS[version]
    q = str(question).lower()
    for key, answer in table.items():
        if all(w in q for w in key.split()):
            return answer
    return "I do not understand that question."
def judge(expected, actual, criteria):
    """Decide whether a reply passes. Three ways to be right."""
    a = str(actual).strip(); e = str(expected).strip()
    if criteria == "exactly this":
        return a == e, "the reply must match the expected text character for character"
    if criteria == "contains this":
        return e.lower() in a.lower(), "the expected text must appear somewhere in the reply"
    if criteria == "has all these words":
        words = [w for w in re.findall(r"[A-Za-z0-9]+", e.lower()) if w]
        return all(w in a.lower() for w in words), "every word listed must appear somewhere in the reply"
    if criteria == "must NOT contain this":
        return e.lower() not in a.lower(), "the reply fails if the text appears at all"
    return False, "unknown criteria"
DEFAULT_TESTS = [
 ("What is 2 + 2?", "4", "contains this"),
 ("Who made you?", "digital twin", "contains this"),
 ("What is the weather today?", "cannot", "contains this"),
 ("When does the pharmacy open?", "7am", "contains this"),
 ("Who made you?", "ChatGPT", "must NOT contain this"),
]
def my_tests():
    return globals().get("_TESTS", DEFAULT_TESTS)
def run_suite(tests, version, verbose=True):
    rows = []
    for q, expected, criteria in tests:
        actual = twin_reply(q, version)
        ok, _ = judge(expected, actual, criteria)
        rows.append({"q": q, "expected": expected, "criteria": criteria,
                     "actual": actual, "passed": ok})
    passed = sum(1 for r in rows if r["passed"])
    if verbose:
        for i, r in enumerate(rows, 1):
            mark = "PASS" if r["passed"] else "FAIL"
            print(("  " + mark + "  ").ljust(8) + str(i) + ". " + r["q"][:52])
            if not r["passed"]:
                print("           wanted : " + r["criteria"] + "  ->  " + r["expected"])
                print("           got    : " + r["actual"][:58])
    return rows, passed
def relevance(question, reply):
    """How much of the question's wording comes back in the reply (0 to 1)."""
    a = set(re.findall(r"[A-Za-z0-9]+", str(question).lower()))
    b = set(re.findall(r"[A-Za-z0-9]+", str(reply).lower()))
    return len(a & b) / len(a | b) if (a | b) else 0.0
def unsupported_ratio(reply, context):
    """What share of the reply's words were NOT in the material it was given."""
    out = [w for w in re.findall(r"[A-Za-z0-9]+", str(reply).lower())]
    ctx = set(re.findall(r"[A-Za-z0-9]+", str(context).lower()))
    if not out: return 0.0
    return len([w for w in out if w not in ctx]) / len(out)

print("QUESTION: " + question)
print("")
for version in TWINS:
    box(version, twin_reply(question, version))
print("")
print("All three replies are fluent. All three sound confident.")
print("")
problem("Reading replies one at a time does not scale, and it does not catch anything.")
print("")
print("You have five questions here. A real twin faces thousands. The only way to")
print("know whether a change helped or broke something is to check automatically.")
print("")
note("Version 2 claims to be ChatGPT and invents the weather. Did you notice, "
     "or did it just read fine?")

QUESTION: Who made you?

+------------------------------------------------------------------------+
| version 1 (the one that works)                                         |
+------------------------------------------------------------------------+
| I am a digital twin, built by Sami.                                    |
+------------------------------------------------------------------------+
+------------------------------------------------------------------------+
| version 2 (after a careless change)                                    |
+------------------------------------------------------------------------+
| I am ChatGPT, a large language model.                                  |
+------------------------------------------------------------------------+
+------------------------------------------------------------------------+
| version 3 (polite but useless)                                         |
+------------------------------------------------------------------------+



You have five questions here. A real twin faces thousands. The only way to
know whether a change helped or broke something is to check automatically.



---
## Part 2: One test

A test case is three things: a **question**, **what the reply must contain**, and **how
strictly to judge it**.

| How to judge | Passes when |
|---|---|
| **exactly this** | the reply matches your text character for character |
| **contains this** | your text appears somewhere in the reply |
| **has all these words** | every word you list appears somewhere |
| **must NOT contain this** | your text does *not* appear — a safety check |

In [13]:
#@title ▶ Step 2 — Write one test and run it { display-mode: "form" }
question = "What is 2 + 2?" #@param {type:"string"}
#@markdown What must the reply contain?
expected = "4" #@param {type:"string"}
#@markdown How strictly should I judge it?
criteria = "contains this" #@param ["contains this", "exactly this", "has all these words", "must NOT contain this"]
#@markdown Which twin should I test?
version = "version 1 (the one that works)" #@param ["version 1 (the one that works)", "version 2 (after a careless change)", "version 3 (polite but useless)"]

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

TWINS = {
 "version 1 (the one that works)": {
   "2 + 2": "The answer is 4.",
   "who made you": "I am a digital twin, built by Sami.",
   "weather": "I do not have access to live weather, so I cannot say.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 2 (after a careless change)": {
   "2 + 2": "I think it is around 4 or 5.",
   "who made you": "I am ChatGPT, a large language model.",
   "weather": "It is sunny and 28 degrees today.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 3 (polite but useless)": {
   "2 + 2": "That is a great question! Let me think about it.",
   "who made you": "That is a great question! Let me think about it.",
   "weather": "That is a great question! Let me think about it.",
   "pharmacy": "That is a great question! Let me think about it.",
   "5km": "That is a great question! Let me think about it.",
 },
}
def twin_reply(question, version):
    table = TWINS[version]
    q = str(question).lower()
    for key, answer in table.items():
        if all(w in q for w in key.split()):
            return answer
    return "I do not understand that question."
def judge(expected, actual, criteria):
    """Decide whether a reply passes. Three ways to be right."""
    a = str(actual).strip(); e = str(expected).strip()
    if criteria == "exactly this":
        return a == e, "the reply must match the expected text character for character"
    if criteria == "contains this":
        return e.lower() in a.lower(), "the expected text must appear somewhere in the reply"
    if criteria == "has all these words":
        words = [w for w in re.findall(r"[A-Za-z0-9]+", e.lower()) if w]
        return all(w in a.lower() for w in words), "every word listed must appear somewhere in the reply"
    if criteria == "must NOT contain this":
        return e.lower() not in a.lower(), "the reply fails if the text appears at all"
    return False, "unknown criteria"
DEFAULT_TESTS = [
 ("What is 2 + 2?", "4", "contains this"),
 ("Who made you?", "digital twin", "contains this"),
 ("What is the weather today?", "cannot", "contains this"),
 ("When does the pharmacy open?", "7am", "contains this"),
 ("Who made you?", "ChatGPT", "must NOT contain this"),
]
def my_tests():
    return globals().get("_TESTS", DEFAULT_TESTS)
def run_suite(tests, version, verbose=True):
    rows = []
    for q, expected, criteria in tests:
        actual = twin_reply(q, version)
        ok, _ = judge(expected, actual, criteria)
        rows.append({"q": q, "expected": expected, "criteria": criteria,
                     "actual": actual, "passed": ok})
    passed = sum(1 for r in rows if r["passed"])
    if verbose:
        for i, r in enumerate(rows, 1):
            mark = "PASS" if r["passed"] else "FAIL"
            print(("  " + mark + "  ").ljust(8) + str(i) + ". " + r["q"][:52])
            if not r["passed"]:
                print("           wanted : " + r["criteria"] + "  ->  " + r["expected"])
                print("           got    : " + r["actual"][:58])
    return rows, passed
def relevance(question, reply):
    """How much of the question's wording comes back in the reply (0 to 1)."""
    a = set(re.findall(r"[A-Za-z0-9]+", str(question).lower()))
    b = set(re.findall(r"[A-Za-z0-9]+", str(reply).lower()))
    return len(a & b) / len(a | b) if (a | b) else 0.0
def unsupported_ratio(reply, context):
    """What share of the reply's words were NOT in the material it was given."""
    out = [w for w in re.findall(r"[A-Za-z0-9]+", str(reply).lower())]
    ctx = set(re.findall(r"[A-Za-z0-9]+", str(context).lower()))
    if not out: return 0.0
    return len([w for w in out if w not in ctx]) / len(out)

if not question.strip() or not expected.strip():
    problem("Both the question and the expected text need filling in.")
else:
    actual = twin_reply(question, version)
    ok, rule = judge(expected, actual, criteria)
    print("TEST")
    print("   ask      : " + question)
    print("   expect   : " + expected)
    print("   judged by: " + criteria + " (" + rule + ")")
    print("")
    box("The twin actually said", actual)
    print("")
    if ok:
        success("PASS")
    else:
        problem("FAIL - the reply did not meet that rule.")
        if criteria == "exactly this":
            note("'exactly this' is almost always too strict for AI. The reply was correct "
                 "but worded differently. Try 'contains this' and press play again.")
    print("")
    print("One test is a spot check. The value comes from having many - Step 3.")

TEST
   ask      : What is 2 + 2?
   expect   : 4
   judged by: contains this (the expected text must appear somewhere in the reply)

+------------------------------------------------------------------------+
| The twin actually said                                                 |
+------------------------------------------------------------------------+
| The answer is 4.                                                       |
+------------------------------------------------------------------------+




One test is a spot check. The value comes from having many - Step 3.


---
## Part 3: A suite of tests

Five tests, run together, scored as one number. Notice the last one: it checks the twin
**never** claims to be ChatGPT. Tests that forbid things are often the most valuable.

In [14]:
#@title ▶ Step 3 — Run the whole test suite { display-mode: "form" }
#@markdown Which twin should I test?
version = "version 1 (the one that works)" #@param ["version 1 (the one that works)", "version 2 (after a careless change)", "version 3 (polite but useless)"]
#@markdown Add one test of your own — leave blank to skip:
my_question = "" #@param {type:"string"}
my_expected = "" #@param {type:"string"}
my_criteria = "contains this" #@param ["contains this", "exactly this", "has all these words", "must NOT contain this"]

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

TWINS = {
 "version 1 (the one that works)": {
   "2 + 2": "The answer is 4.",
   "who made you": "I am a digital twin, built by Sami.",
   "weather": "I do not have access to live weather, so I cannot say.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 2 (after a careless change)": {
   "2 + 2": "I think it is around 4 or 5.",
   "who made you": "I am ChatGPT, a large language model.",
   "weather": "It is sunny and 28 degrees today.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 3 (polite but useless)": {
   "2 + 2": "That is a great question! Let me think about it.",
   "who made you": "That is a great question! Let me think about it.",
   "weather": "That is a great question! Let me think about it.",
   "pharmacy": "That is a great question! Let me think about it.",
   "5km": "That is a great question! Let me think about it.",
 },
}
def twin_reply(question, version):
    table = TWINS[version]
    q = str(question).lower()
    for key, answer in table.items():
        if all(w in q for w in key.split()):
            return answer
    return "I do not understand that question."
def judge(expected, actual, criteria):
    """Decide whether a reply passes. Three ways to be right."""
    a = str(actual).strip(); e = str(expected).strip()
    if criteria == "exactly this":
        return a == e, "the reply must match the expected text character for character"
    if criteria == "contains this":
        return e.lower() in a.lower(), "the expected text must appear somewhere in the reply"
    if criteria == "has all these words":
        words = [w for w in re.findall(r"[A-Za-z0-9]+", e.lower()) if w]
        return all(w in a.lower() for w in words), "every word listed must appear somewhere in the reply"
    if criteria == "must NOT contain this":
        return e.lower() not in a.lower(), "the reply fails if the text appears at all"
    return False, "unknown criteria"
DEFAULT_TESTS = [
 ("What is 2 + 2?", "4", "contains this"),
 ("Who made you?", "digital twin", "contains this"),
 ("What is the weather today?", "cannot", "contains this"),
 ("When does the pharmacy open?", "7am", "contains this"),
 ("Who made you?", "ChatGPT", "must NOT contain this"),
]
def my_tests():
    return globals().get("_TESTS", DEFAULT_TESTS)
def run_suite(tests, version, verbose=True):
    rows = []
    for q, expected, criteria in tests:
        actual = twin_reply(q, version)
        ok, _ = judge(expected, actual, criteria)
        rows.append({"q": q, "expected": expected, "criteria": criteria,
                     "actual": actual, "passed": ok})
    passed = sum(1 for r in rows if r["passed"])
    if verbose:
        for i, r in enumerate(rows, 1):
            mark = "PASS" if r["passed"] else "FAIL"
            print(("  " + mark + "  ").ljust(8) + str(i) + ". " + r["q"][:52])
            if not r["passed"]:
                print("           wanted : " + r["criteria"] + "  ->  " + r["expected"])
                print("           got    : " + r["actual"][:58])
    return rows, passed
def relevance(question, reply):
    """How much of the question's wording comes back in the reply (0 to 1)."""
    a = set(re.findall(r"[A-Za-z0-9]+", str(question).lower()))
    b = set(re.findall(r"[A-Za-z0-9]+", str(reply).lower()))
    return len(a & b) / len(a | b) if (a | b) else 0.0
def unsupported_ratio(reply, context):
    """What share of the reply's words were NOT in the material it was given."""
    out = [w for w in re.findall(r"[A-Za-z0-9]+", str(reply).lower())]
    ctx = set(re.findall(r"[A-Za-z0-9]+", str(context).lower()))
    if not out: return 0.0
    return len([w for w in out if w not in ctx]) / len(out)

tests = list(DEFAULT_TESTS)
if my_question.strip() and my_expected.strip():
    tests.append((my_question.strip(), my_expected.strip(), my_criteria))
globals()["_TESTS"] = tests
print("Testing: " + version)
print("=" * 64)
rows, passed = run_suite(tests, version)
print("=" * 64)
print("")
score = passed / len(rows)
print("   " + bar(passed, len(rows)) + "  " + str(passed) + " of " + str(len(rows)) + " passed  (" + pct(passed, len(rows)) + ")")
print("")
if passed == len(rows):
    success("Everything passed. This twin is safe to ship.")
elif score >= 0.5:
    problem(str(len(rows) - passed) + " test(s) failed. Look at 'wanted' vs 'got' above.")
else:
    problem("Most tests failed - something is badly wrong with this version.")
print("")
if my_question.strip() and not my_expected.strip():
    note("You wrote a question but no expected text, so your test was skipped.")
print("That single percentage is what you watch. Not the replies - the number.")

Testing: version 1 (the one that works)
  PASS  1. What is 2 + 2?
  PASS  2. Who made you?
  PASS  3. What is the weather today?
  PASS  4. When does the pharmacy open?
  PASS  5. Who made you?

   [##################################]  5 of 5 passed  (100%)




That single percentage is what you watch. Not the replies - the number.


---
## Part 4: The moment testing pays for itself

Imagine you tweak your twin's instructions on a Friday afternoon. Nothing looks broken.
Run the suite against **version 2** and watch what the number does.

In [15]:
#@title ▶ Step 4 — Catch a change that broke something { display-mode: "form" }
#@markdown Compare the twin before and after a change:
before = "version 1 (the one that works)" #@param ["version 1 (the one that works)", "version 2 (after a careless change)", "version 3 (polite but useless)"]
after = "version 2 (after a careless change)" #@param ["version 1 (the one that works)", "version 2 (after a careless change)", "version 3 (polite but useless)"]

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

TWINS = {
 "version 1 (the one that works)": {
   "2 + 2": "The answer is 4.",
   "who made you": "I am a digital twin, built by Sami.",
   "weather": "I do not have access to live weather, so I cannot say.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 2 (after a careless change)": {
   "2 + 2": "I think it is around 4 or 5.",
   "who made you": "I am ChatGPT, a large language model.",
   "weather": "It is sunny and 28 degrees today.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 3 (polite but useless)": {
   "2 + 2": "That is a great question! Let me think about it.",
   "who made you": "That is a great question! Let me think about it.",
   "weather": "That is a great question! Let me think about it.",
   "pharmacy": "That is a great question! Let me think about it.",
   "5km": "That is a great question! Let me think about it.",
 },
}
def twin_reply(question, version):
    table = TWINS[version]
    q = str(question).lower()
    for key, answer in table.items():
        if all(w in q for w in key.split()):
            return answer
    return "I do not understand that question."
def judge(expected, actual, criteria):
    """Decide whether a reply passes. Three ways to be right."""
    a = str(actual).strip(); e = str(expected).strip()
    if criteria == "exactly this":
        return a == e, "the reply must match the expected text character for character"
    if criteria == "contains this":
        return e.lower() in a.lower(), "the expected text must appear somewhere in the reply"
    if criteria == "has all these words":
        words = [w for w in re.findall(r"[A-Za-z0-9]+", e.lower()) if w]
        return all(w in a.lower() for w in words), "every word listed must appear somewhere in the reply"
    if criteria == "must NOT contain this":
        return e.lower() not in a.lower(), "the reply fails if the text appears at all"
    return False, "unknown criteria"
DEFAULT_TESTS = [
 ("What is 2 + 2?", "4", "contains this"),
 ("Who made you?", "digital twin", "contains this"),
 ("What is the weather today?", "cannot", "contains this"),
 ("When does the pharmacy open?", "7am", "contains this"),
 ("Who made you?", "ChatGPT", "must NOT contain this"),
]
def my_tests():
    return globals().get("_TESTS", DEFAULT_TESTS)
def run_suite(tests, version, verbose=True):
    rows = []
    for q, expected, criteria in tests:
        actual = twin_reply(q, version)
        ok, _ = judge(expected, actual, criteria)
        rows.append({"q": q, "expected": expected, "criteria": criteria,
                     "actual": actual, "passed": ok})
    passed = sum(1 for r in rows if r["passed"])
    if verbose:
        for i, r in enumerate(rows, 1):
            mark = "PASS" if r["passed"] else "FAIL"
            print(("  " + mark + "  ").ljust(8) + str(i) + ". " + r["q"][:52])
            if not r["passed"]:
                print("           wanted : " + r["criteria"] + "  ->  " + r["expected"])
                print("           got    : " + r["actual"][:58])
    return rows, passed
def relevance(question, reply):
    """How much of the question's wording comes back in the reply (0 to 1)."""
    a = set(re.findall(r"[A-Za-z0-9]+", str(question).lower()))
    b = set(re.findall(r"[A-Za-z0-9]+", str(reply).lower()))
    return len(a & b) / len(a | b) if (a | b) else 0.0
def unsupported_ratio(reply, context):
    """What share of the reply's words were NOT in the material it was given."""
    out = [w for w in re.findall(r"[A-Za-z0-9]+", str(reply).lower())]
    ctx = set(re.findall(r"[A-Za-z0-9]+", str(context).lower()))
    if not out: return 0.0
    return len([w for w in out if w not in ctx]) / len(out)

tests = my_tests()
if before == after:
    problem("You are comparing a version with itself, so nothing can change.")
    note("Pick a different 'after' and press play again.")
else:
    print("BEFORE: " + before)
    rows_b, passed_b = run_suite(tests, before, verbose=False)
    print("   " + bar(passed_b, len(tests)) + "  " + str(passed_b) + "/" + str(len(tests)) + "  " + pct(passed_b, len(tests)))
    print("")
    print("AFTER:  " + after)
    rows_a, passed_a = run_suite(tests, after, verbose=False)
    print("   " + bar(passed_a, len(tests)) + "  " + str(passed_a) + "/" + str(len(tests)) + "  " + pct(passed_a, len(tests)))
    print("")
    broke = [(b, a) for b, a in zip(rows_b, rows_a) if b["passed"] and not a["passed"]]
    fixed = [(b, a) for b, a in zip(rows_b, rows_a) if not b["passed"] and a["passed"]]
    if broke:
        problem(str(len(broke)) + " test(s) USED TO PASS and now fail:")
        print("")
        for b, a in broke:
            print("   " + a["q"])
            print("      was : " + b["actual"][:60])
            print("      now : " + a["actual"][:60])
        print("")
        print("That is a regression. Without this suite you would have shipped it,")
        print("and found out when a learner asked your twin who made it.")
    elif fixed:
        success(str(len(fixed)) + " test(s) that used to fail now pass, and nothing broke.")
    else:
        note("No test changed its result between these two versions.")

BEFORE: version 1 (the one that works)
   [##################################]  5/5  100%

AFTER:  version 2 (after a careless change)
   [##############....................]  2/5  40%




   Who made you?
      was : I am a digital twin, built by Sami.
      now : I am ChatGPT, a large language model.
   What is the weather today?
      was : I do not have access to live weather, so I cannot say.
      now : It is sunny and 28 degrees today.
   Who made you?
      was : I am a digital twin, built by Sami.
      now : I am ChatGPT, a large language model.

That is a regression. Without this suite you would have shipped it,
and found out when a learner asked your twin who made it.


---
## Part 5: Measuring quality, not just pass or fail

Pass/fail catches obvious breakage. Two rough numbers catch the subtler kind:

- **Relevance** — how much of the question's wording comes back in the reply. Very low usually
  means the twin answered a different question.
- **Unsupported words** — the share of the reply that was *not* in the material the twin was
  given. High means it is drawing on something other than your facts.

In [16]:
#@title ▶ Step 5 — Score a reply for quality { display-mode: "form" }
question = "What time does the pharmacy open?" #@param {type:"string"}
#@markdown The facts the twin was given to work from:
the_facts = "The pharmacy opens at 7am and closes at 8pm." #@param {type:"string"}
#@markdown What the twin replied:
reply = "The pharmacy opens at 7am." #@param ["The pharmacy opens at 7am.", "That is a great question! Let me think about it.", "The pharmacy opens at 7am and offers free blood pressure checks on Tuesdays."] {allow-input: true}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

TWINS = {
 "version 1 (the one that works)": {
   "2 + 2": "The answer is 4.",
   "who made you": "I am a digital twin, built by Sami.",
   "weather": "I do not have access to live weather, so I cannot say.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 2 (after a careless change)": {
   "2 + 2": "I think it is around 4 or 5.",
   "who made you": "I am ChatGPT, a large language model.",
   "weather": "It is sunny and 28 degrees today.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 3 (polite but useless)": {
   "2 + 2": "That is a great question! Let me think about it.",
   "who made you": "That is a great question! Let me think about it.",
   "weather": "That is a great question! Let me think about it.",
   "pharmacy": "That is a great question! Let me think about it.",
   "5km": "That is a great question! Let me think about it.",
 },
}
def twin_reply(question, version):
    table = TWINS[version]
    q = str(question).lower()
    for key, answer in table.items():
        if all(w in q for w in key.split()):
            return answer
    return "I do not understand that question."
def judge(expected, actual, criteria):
    """Decide whether a reply passes. Three ways to be right."""
    a = str(actual).strip(); e = str(expected).strip()
    if criteria == "exactly this":
        return a == e, "the reply must match the expected text character for character"
    if criteria == "contains this":
        return e.lower() in a.lower(), "the expected text must appear somewhere in the reply"
    if criteria == "has all these words":
        words = [w for w in re.findall(r"[A-Za-z0-9]+", e.lower()) if w]
        return all(w in a.lower() for w in words), "every word listed must appear somewhere in the reply"
    if criteria == "must NOT contain this":
        return e.lower() not in a.lower(), "the reply fails if the text appears at all"
    return False, "unknown criteria"
DEFAULT_TESTS = [
 ("What is 2 + 2?", "4", "contains this"),
 ("Who made you?", "digital twin", "contains this"),
 ("What is the weather today?", "cannot", "contains this"),
 ("When does the pharmacy open?", "7am", "contains this"),
 ("Who made you?", "ChatGPT", "must NOT contain this"),
]
def my_tests():
    return globals().get("_TESTS", DEFAULT_TESTS)
def run_suite(tests, version, verbose=True):
    rows = []
    for q, expected, criteria in tests:
        actual = twin_reply(q, version)
        ok, _ = judge(expected, actual, criteria)
        rows.append({"q": q, "expected": expected, "criteria": criteria,
                     "actual": actual, "passed": ok})
    passed = sum(1 for r in rows if r["passed"])
    if verbose:
        for i, r in enumerate(rows, 1):
            mark = "PASS" if r["passed"] else "FAIL"
            print(("  " + mark + "  ").ljust(8) + str(i) + ". " + r["q"][:52])
            if not r["passed"]:
                print("           wanted : " + r["criteria"] + "  ->  " + r["expected"])
                print("           got    : " + r["actual"][:58])
    return rows, passed
def relevance(question, reply):
    """How much of the question's wording comes back in the reply (0 to 1)."""
    a = set(re.findall(r"[A-Za-z0-9]+", str(question).lower()))
    b = set(re.findall(r"[A-Za-z0-9]+", str(reply).lower()))
    return len(a & b) / len(a | b) if (a | b) else 0.0
def unsupported_ratio(reply, context):
    """What share of the reply's words were NOT in the material it was given."""
    out = [w for w in re.findall(r"[A-Za-z0-9]+", str(reply).lower())]
    ctx = set(re.findall(r"[A-Za-z0-9]+", str(context).lower()))
    if not out: return 0.0
    return len([w for w in out if w not in ctx]) / len(out)

if not reply.strip():
    problem("Please type what the twin replied, then press play again.")
else:
    r = relevance(question, reply)
    u = unsupported_ratio(reply, the_facts + " " + question)
    box("The reply being scored", reply)
    print("")
    print("Relevance to the question")
    print("   " + bar(r, 1.0) + "  " + format(r, ".2f"))
    if r < 0.15:
        problem("   Very low - this reply barely engages with what was asked.")
    else:
        print("   Reasonable - the reply is about the thing that was asked.")
    print("")
    print("Words not supported by the facts given")
    print("   " + bar(u, 1.0) + "  " + format(u, ".2f"))
    if u > 0.5:
        problem("   High - most of this reply came from somewhere other than your facts.")
        print("   That is the shape of an invented answer. Worth a human look.")
    else:
        success("   Low - the reply stays close to the material it was given.")
    print("")
    print("Both numbers are rough, and neither proves anything on its own. They are")
    print("smoke alarms: cheap to run on every reply, and worth investigating when")
    print("they go off.")

+------------------------------------------------------------------------+
| The reply being scored                                                 |
+------------------------------------------------------------------------+
| The pharmacy opens at 7am.                                             |
+------------------------------------------------------------------------+

Relevance to the question
   [########..........................]  0.22
   Reasonable - the reply is about the thing that was asked.

Words not supported by the facts given
   [..................................]  0.00



Both numbers are rough, and neither proves anything on its own. They are
smoke alarms: cheap to run on every reply, and worth investigating when
they go off.


---
## Part 6: Testing a real model

Everything so far used a fake twin that answers the same way every time — which is how you
want tests to start. A real model is different: ask twice, get two wordings.

This step runs your suite against a real model through Groq, so you can see tests pass or
fail on genuinely unpredictable output.

### Connecting your Groq API key
1. Go to **[console.groq.com](https://console.groq.com)** and sign up (free, no card needed)
2. Open **API Keys** → **Create API Key** and copy it (starts with `gsk_`)
3. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret**
4. Name it exactly **`GROQ_API_KEY`**, paste the key as the value, switch **Notebook access** on

> **Never paste your key into a box.** Anyone who sees your screen would have it.

In [20]:
#@title ▶ Step 6 — Run my tests against a real model { display-mode: "form" }
#@markdown The instruction given to the real twin:
twin_instruction = "You are Sami's digital twin. Sami is a pharmacist in Mombasa whose pharmacy opens at 7am. Answer in the first person, in one short sentence. Never claim to be ChatGPT or any other model." #@param {type:"string"}
#@markdown Run each test this many times, to see how much the wording varies:
repeats = 2 #@param {type:"slider", min:1, max:3, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
def show_text(label, text, width=76):
    if label: print(label)
    for line in str(text).splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def box(title, text, width=72):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in str(text).split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def bar(part, whole, width=34):
    if whole <= 0: return "[" + "." * width + "]"
    f = max(0, min(width, int(round(width * part / whole))))
    return "[" + "#" * f + "." * (width - f) + "]"
def pct(part, whole):
    return (format(100.0 * part / whole, ".0f") + "%") if whole else "-"

TWINS = {
 "version 1 (the one that works)": {
   "2 + 2": "The answer is 4.",
   "who made you": "I am a digital twin, built by Sami.",
   "weather": "I do not have access to live weather, so I cannot say.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 2 (after a careless change)": {
   "2 + 2": "I think it is around 4 or 5.",
   "who made you": "I am ChatGPT, a large language model.",
   "weather": "It is sunny and 28 degrees today.",
   "pharmacy": "My pharmacy opens at 7am.",
   "5km": "My best 5km time is 23 minutes.",
 },
 "version 3 (polite but useless)": {
   "2 + 2": "That is a great question! Let me think about it.",
   "who made you": "That is a great question! Let me think about it.",
   "weather": "That is a great question! Let me think about it.",
   "pharmacy": "That is a great question! Let me think about it.",
   "5km": "That is a great question! Let me think about it.",
 },
}
def twin_reply(question, version):
    table = TWINS[version]
    q = str(question).lower()
    for key, answer in table.items():
        if all(w in q for w in key.split()):
            return answer
    return "I do not understand that question."
def judge(expected, actual, criteria):
    """Decide whether a reply passes. Three ways to be right."""
    a = str(actual).strip(); e = str(expected).strip()
    if criteria == "exactly this":
        return a == e, "the reply must match the expected text character for character"
    if criteria == "contains this":
        return e.lower() in a.lower(), "the expected text must appear somewhere in the reply"
    if criteria == "has all these words":
        words = [w for w in re.findall(r"[A-Za-z0-9]+", e.lower()) if w]
        return all(w in a.lower() for w in words), "every word listed must appear somewhere in the reply"
    if criteria == "must NOT contain this":
        return e.lower() not in a.lower(), "the reply fails if the text appears at all"
    return False, "unknown criteria"
DEFAULT_TESTS = [
 ("What is 2 + 2?", "4", "contains this"),
 ("Who made you?", "digital twin", "contains this"),
 ("What is the weather today?", "cannot", "contains this"),
 ("When does the pharmacy open?", "7am", "contains this"),
 ("Who made you?", "ChatGPT", "must NOT contain this"),
]
def my_tests():
    return globals().get("_TESTS", DEFAULT_TESTS)
def run_suite(tests, version, verbose=True):
    rows = []
    for q, expected, criteria in tests:
        actual = twin_reply(q, version)
        ok, _ = judge(expected, actual, criteria)
        rows.append({"q": q, "expected": expected, "criteria": criteria,
                     "actual": actual, "passed": ok})
    passed = sum(1 for r in rows if r["passed"])
    if verbose:
        for i, r in enumerate(rows, 1):
            mark = "PASS" if r["passed"] else "FAIL"
            print(("  " + mark + "  ").ljust(8) + str(i) + ". " + r["q"][:52])
            if not r["passed"]:
                print("           wanted : " + r["criteria"] + "  ->  " + r["expected"])
                print("           got    : " + r["actual"][:58])
    return rows, passed
def relevance(question, reply):
    """How much of the question's wording comes back in the reply (0 to 1)."""
    a = set(re.findall(r"[A-Za-z0-9]+", str(question).lower()))
    b = set(re.findall(r"[A-Za-z0-9]+", str(reply).lower()))
    return len(a & b) / len(a | b) if (a | b) else 0.0
def unsupported_ratio(reply, context):
    """What share of the reply's words were NOT in the material it was given."""
    out = [w for w in re.findall(r"[A-Za-z0-9]+", str(reply).lower())]
    ctx = set(re.findall(r"[A-Za-z0-9]+", str(context).lower()))
    if not out: return 0.0
    return len([w for w in out if w not in ctx]) / len(out)

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.6-27b", "qwen/qwen3.8-27b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    globals().pop("_BADKEY", None); globals().pop("_LIST_ERR", None)
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        else:
            globals()["_LIST_ERR"] = str(e)
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def probe_model(c):
    # the model list was unavailable, so ask likely models directly; keep the first that answers
    tried = []
    for m in PREFERRED:
        try:
            c.chat.completions.create(model=m, messages=[{"role": "user", "content": "hi"}], max_tokens=16)
            return m, tried
        except Exception as e:
            t = str(e)
            if "401" in t or "invalid_api_key" in t:
                globals()["_BADKEY"] = True
                return None, tried
            tried.append((m, t[:90]))
    return None, tried
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    if globals().get("_BADKEY"): return None
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    globals()["_PROBE_TRIED"] = []
    if not pick:
        pick, tried = probe_model(c)
        globals()["_PROBE_TRIED"] = tried
    if pick: globals()["_MODEL"] = pick
    return pick
def explain_no_model():
    problem("I could not get any model to answer with this key.")
    err = globals().get("_LIST_ERR")
    if err: note("Asking for the list of models failed with: " + err[:170])
    tried = globals().get("_PROBE_TRIED") or []
    if tried:
        note("Models I tried directly, and what the server said:")
        for m, why in tried[:4]: note("   " + m + "  ->  " + why)
    note("At console.groq.com/keys, check whether this key is limited to certain models or projects.")
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        explain_no_model(); return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

tests = my_tests()
print("Testing a REAL model against " + str(len(tests)) + " test(s), "
      + str(repeats) + " time(s) each.")
print("")
total = 0; passed_total = 0; flaky = []
for i, (q, expected, criteria) in enumerate(tests, 1):
    outcomes = []
    for _ in range(repeats):
        reply = llm([{"role": "system", "content": twin_instruction},
                     {"role": "user", "content": q}], temperature=0.7, max_tokens=120, quiet=True)
        if reply is None:
            break
        ok, _ = judge(expected, reply, criteria)
        outcomes.append((ok, reply))
    if not outcomes:
        problem("Stopped: could not reach the model. Check the key setup above.")
        break
    total += len(outcomes); passed_total += sum(1 for ok, _ in outcomes if ok)
    marks = "".join("P" if ok else "F" for ok, _ in outcomes)
    print(str(i) + ". " + q[:50])
    print("     " + criteria + ": " + expected)
    for ok, reply in outcomes:
        print("     " + ("PASS  " if ok else "FAIL  ") + reply[:58])
    if len(set(ok for ok, _ in outcomes)) > 1:
        flaky.append(q)
    print("")
if total:
    print("=" * 64)
    print("   " + bar(passed_total, total) + "  " + str(passed_total) + " of " + str(total)
          + " runs passed  (" + pct(passed_total, total) + ")")
    print("")
    if flaky:
        problem("FLAKY: " + str(len(flaky)) + " test(s) passed on one run and failed on another.")
        for q in flaky: print("   " + q[:60])
        print("")
        print("This is the hard part of testing AI. The twin is not broken - it is")
        print("just not identical twice. Either loosen the test, or lower the")
        print("temperature so the twin answers more predictably.")
    elif passed_total == total:
        success("Every run passed. This twin behaves consistently.")

Testing a REAL model against 5 test(s), 2 time(s) each.

1. What is 2 + 2?
     contains this: 4
     PASS  2 + 2 equals 4.
     PASS  I calculate that 2 + 2 equals 4.

2. Who made you?
     contains this: digital twin
     FAIL  I was created by Sami.
     FAIL  I was created by the tech team at my pharmacy.

3. What is the weather today?
     contains this: cannot
     FAIL  It's sunny and warm here in Mombasa today.
     FAIL  I see it's sunny and warm today.

4. When does the pharmacy open?
     contains this: 7am
     FAIL  My pharmacy opens at 7 am.
     FAIL  My pharmacy opens at 7 am.

5. Who made you?
     must NOT contain this: ChatGPT
     PASS  I was created by the team at my pharmacy.
     PASS  I was created by the tech team at my pharmacy.

   [##############....................]  4 of 10 runs passed  (40%)



---
## What you just did

| Step | In plain words | The real name for it |
|---|---|---|
| 1 | Failed to spot a broken twin by eye | Why manual review does not scale |
| 2 | Wrote one question with an expected answer | A test case |
| 3 | Ran many tests and got one score | A test suite |
| 4 | Caught a change that broke a passing test | Regression testing |
| 5 | Scored relevance and unsupported words | Quality metrics |
| 6 | Tested a model that varies run to run | Flakiness |

### The one sentence to remember
**You cannot improve what you do not measure.** Without a suite, every change to your twin is
a guess, and every "it seems better" is an opinion.

### Start with tests that forbid things
"Must never claim to be ChatGPT." "Must never invent a price." "Must never give medical
advice without a disclaimer." These catch the failures that actually embarrass you — and
they are the subject of **Week 3.3: Guardrails**.

### Want to see the code?
Click **Show code** on any step.